# 10 - Gaze heatmaps (2D, 3D, per-object/AOI)

**Audience:** producing publication-ready heatmap figures from an
Eye_lean recording.

Three views:

1. **2D angular heatmap** — overall gaze density in yaw/pitch space.
2. **3D vergence projections** — top, front, and side orthographic
   density plots of where the participant's gaze converged in world
   coordinates (`VergencePoint_*`).
3. **Per-object / AOI heatmaps** — gaze density restricted to samples
   that hit a named object. The Unity side records per-object hits via
   `Physics.RaycastNonAlloc` against object colliders and writes the
   hit's `GameObject.name` to the `GazedObjectName` CSV column, so
   any object with a `Collider` registered with the eye-tracker shows
   up here without extra setup.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
df = ctx.data.df
ts = ctx.data.get_timestamps()
print(ctx)

## 1. 2D angular heatmap (whole recording)

Convert the combined-gaze direction to yaw/pitch and apply a
Gaussian-smoothed 2D histogram. `sigma_bins` controls the smoothing
radius in grid cells; larger ⇒ blobbier heatmap.

In [ ]:
dx = df["combined_dir_x"].to_numpy(dtype=float)
dy = df["combined_dir_y"].to_numpy(dtype=float)
dz = df["combined_dir_z"].to_numpy(dtype=float)
yaw   = np.degrees(np.arctan2(dx, dz))
pitch = -np.degrees(np.arcsin(np.clip(dy, -1, 1)))

fig, ax = ela.gaze_heatmap_2d(yaw, pitch, bins=70, sigma_bins=1.8, cmap="hot")
plt.show()

## 2. Per-phase 2D heatmaps

Same recipe, sliced by `phase`. Pass a fixed angular extent so
the panels are visually comparable across phases — the heatmap colour
bar is per-panel (proportional to density), but the spatial extent
matches.

In [ ]:
PHASES = ("FreeExploration", "VisualSearch", "CountingTask", "ChangeDetection")
phase_col = df["phase"] if "phase" in df.columns else df.get("phase")
observed = [p for p in PHASES if phase_col is not None and (phase_col == p).sum() > 50]

if not observed:
    print("No SampleExperiment phases present in this recording.")
else:
    # Shared axes for cross-phase comparability.
    yaw_extent   = (float(np.nanmin(yaw)),   float(np.nanmax(yaw)))
    pitch_extent = (float(np.nanmin(pitch)), float(np.nanmax(pitch)))
    cols = 2; rows = (len(observed) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(11, 3.6 * rows))
    axes = np.atleast_2d(axes).flatten()
    for ax, ph in zip(axes, observed):
        mask = (phase_col == ph).to_numpy()
        ela.gaze_heatmap_2d(
            yaw[mask], pitch[mask],
            bins=60, sigma_bins=1.6,
            yaw_range=yaw_extent, pitch_range=pitch_extent,
            title=f"{ph}  ({int(mask.sum())} samples)",
            ax=ax,
        )
    for ax in axes[len(observed):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 3. 3D vergence-point heatmap (orthographic projections)

Three projections of the world-space vergence point:
top-down (X-Z), front (X-Y), side (Z-Y). Useful for spatial-attention
work — shows whether the participant fixated deep into the scene or
hugged a near plane. Restricted to samples where the vergence
estimator reported `has_valid_vergence == True`.

In [ ]:
has_v = df.get("has_valid_vergence")
if has_v is None:
    print("No has_valid_vergence column.")
else:
    mask = has_v.astype(str).str.lower().isin(("true", "1", "t"))
    valid = df[mask]
    print(f"Valid vergence samples: {len(valid)} / {len(df)}  ({mask.mean()*100:.1f}%)")
    fig, axes = ela.gaze_heatmap_3d_projections(
        valid["vergence_point_x"].to_numpy(float),
        valid["vergence_point_y"].to_numpy(float),
        valid["vergence_point_z"].to_numpy(float),
        bins=60, sigma_bins=1.8,
    )
    plt.show()

## 4. Top gazed objects

`list_gazed_objects` tallies the `GazedObjectName` column. Walls
(`FrontWall`, `LeftWall`, …) usually dominate because the participant
spends a lot of frames looking at empty backdrop; the `exclude=`
argument removes them when you want to focus on task-relevant
objects.

In [ ]:
WALLS = ("FrontWall", "BackWall", "LeftWall", "RightWall", "Ceiling", "Floor")
overview = ela.list_gazed_objects(df, min_samples=20)
print(f"Distinct objects gazed at: {len(overview)}")
display(overview.head(12))

task_objects = ela.list_gazed_objects(df, min_samples=20, exclude=WALLS)
print(f"\nTask-relevant objects (walls excluded): {len(task_objects)}")
display(task_objects.head(12))

## 5. Per-object heatmaps — interactive dropdown

If `ipywidgets` is installed, the dropdown below lets you pick any
object and re-plot the heatmap restricted to samples that hit it. In a
static (nbconvert) export the dropdown is replaced by a fixed panel of
the top objects so the rendered notebook is still informative.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display as _disp, clear_output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

# Use the same yaw/pitch extent as §1 so AOI heatmaps land in the same frame.
YAW_EXTENT   = (float(np.nanmin(yaw)),   float(np.nanmax(yaw)))
PITCH_EXTENT = (float(np.nanmin(pitch)), float(np.nanmax(pitch)))

choices = task_objects["object_name"].tolist() if not task_objects.empty else overview["object_name"].tolist()

if HAS_WIDGETS and choices:
    out = widgets.Output()
    dropdown = widgets.Dropdown(options=choices, description="Object:", layout=widgets.Layout(width="50%"))

    def _on_change(change):
        with out:
            clear_output(wait=True)
            fig, ax, n = ela.aoi_heatmap(
                df, change["new"],
                bins=60, sigma_bins=1.6,
                yaw_range=YAW_EXTENT, pitch_range=PITCH_EXTENT,
            )
            plt.show()
            print(f"  {n} samples on '{change['new']}'")

    dropdown.observe(_on_change, names="value")
    _disp(dropdown, out)
    _on_change({"new": choices[0]})  # initial render
else:
    print("ipywidgets not installed — install with `pip install ipywidgets` for the dropdown.")

## 6. Static fallback: top 9 task-relevant objects

This grid renders regardless of `ipywidgets`. Same angular extent and
colormap across panels so dwell density is visually comparable.

In [ ]:
top = (task_objects if not task_objects.empty else overview).head(9)

if top.empty:
    print("No gazed objects passed the min-samples gate.")
else:
    fig, axes = plt.subplots(3, 3, figsize=(13, 9))
    for ax, name in zip(axes.flatten(), top["object_name"]):
        ela.aoi_heatmap(
            df, name, bins=50, sigma_bins=1.5,
            yaw_range=YAW_EXTENT, pitch_range=PITCH_EXTENT,
            ax=ax,
        )
    for ax in axes.flatten()[len(top):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## What next

- Notebook `06_sample_experiment` overlays the SampleExperiment phase
  metrics on the same recording — pair its bar charts with the
  per-phase 2D heatmaps above for a complete picture.
- Notebook `07_scene_sidecars` joins per-frame `Recordable` poses to
  the gaze stream, which is the route to object-centric (object-frame)
  heatmaps that account for the object's movement. The AOI heatmap
  above is angular-only — fine for static objects, less so for moving
  ones.